## EDA Data Cleaning

In [72]:
import pandas as pd
import numpy as np

In [73]:
raw_path = r"D:\Telco Customer Churn\data\Original_dataset.csv"

df = pd.read_csv(raw_path)

print("Original Shape:", df.shape)
print("\nOriginal Columns:")
print(df.columns.tolist())

Original Shape: (7043, 21)

Original Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [74]:
duplicate_count = df.duplicated().sum()

print("\nDuplicate rows before cleaning:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().copy()

print("Duplicate rows after removal:", df.duplicated().sum())


Duplicate rows before cleaning: 0
Duplicate rows after removal: 0


In [75]:
# Remove leading/trailing spaces from string columns
object_columns = df.select_dtypes(include="object").columns

for column in object_columns:
    df[column] = df[column].str.strip()

C:\Users\yashg\AppData\Local\Temp\ipykernel_8372\1417384670.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns


In [76]:
# Convert empty strings to NaN
df = df.replace(r"^\s*$", np.nan, regex=True)


print("\nMissing values after whitespace/blank cleaning:")
print(df.isnull().sum())


print("\nMissing percentage:")
missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_percentage[missing_percentage > 0].round(2))


Missing values after whitespace/blank cleaning:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Missing percentage:
TotalCharges    0.16
dtype: float64


In [77]:
# Convert TotalCharges to numeric
# Invalid/blank values become NaN
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)


print("\nTotalCharges missing values:")
print(df["TotalCharges"].isnull().sum())


# Check the rows where TotalCharges is missing
totalcharges_missing = df[df["TotalCharges"].isnull()]

print("\nRows with missing TotalCharges:")
print(
    totalcharges_missing[
        ["customerID", "tenure", "TotalCharges"]
    ]
)


TotalCharges missing values:
11

Rows with missing TotalCharges:
      customerID  tenure  TotalCharges
488   4472-LVYGI       0           NaN
753   3115-CZMZD       0           NaN
936   5709-LVOEQ       0           NaN
1082  4367-NUYAO       0           NaN
1340  1371-DWPAZ       0           NaN
3331  7644-OMVMY       0           NaN
3826  3213-VVOLG       0           NaN
4380  2520-SGTTA       0           NaN
5218  2923-ARZLG       0           NaN
6670  4075-WKNIU       0           NaN
6754  2775-SEFEE       0           NaN


In [78]:
if df["TotalCharges"].isnull().sum() > 0:

    print("\nTenure values for missing TotalCharges:")
    print(
        df.loc[
            df["TotalCharges"].isnull(),
            "tenure"
        ].value_counts(dropna=False)
    )


# If all missing TotalCharges belong to tenure = 0,
# then TotalCharges logically represents zero accumulated charges.
missing_totalcharges = df["TotalCharges"].isnull()

if missing_totalcharges.any():

    if (
        df.loc[missing_totalcharges, "tenure"]
        .eq(0)
        .all()
    ):
        print(
            "\nAll missing TotalCharges rows have tenure = 0."
        )
        print(
            "Treating missing TotalCharges as 0."
        )

        df.loc[
            missing_totalcharges,
            "TotalCharges"
        ] = 0

    else:
        print(
            "\nWARNING: Missing TotalCharges are not all "
            "associated with tenure = 0."
        )
        print(
            "These missing values require further investigation."
        )


Tenure values for missing TotalCharges:
tenure
0    11
Name: count, dtype: int64

All missing TotalCharges rows have tenure = 0.
Treating missing TotalCharges as 0.


In [79]:
numeric_columns = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

print("\nInvalid numerical values:")

# Negative values are invalid for these columns
for column in [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]:

    invalid = df[df[column] < 0]

    print(
        f"{column}: {len(invalid)} negative values"
    )


# SeniorCitizen must be 0 or 1
invalid_senior = df[
    ~df["SeniorCitizen"].isin([0, 1])
    & df["SeniorCitizen"].notna()
]

print(
    "SeniorCitizen:",
    len(invalid_senior),
    "invalid values"
)


# Tenure should be between 0 and 72 months
invalid_tenure = df[
    ~df["tenure"].between(0, 72)
    & df["tenure"].notna()
]

print(
    "Tenure outside 0-72:",
    len(invalid_tenure)
)


Invalid numerical values:
tenure: 0 negative values
MonthlyCharges: 0 negative values
TotalCharges: 0 negative values
SeniorCitizen: 0 invalid values
Tenure outside 0-72: 0


In [80]:
expected_categories = {

    "gender": {
        "Female",
        "Male"
    },

    "Partner": {
        "Yes",
        "No"
    },

    "Dependents": {
        "Yes",
        "No"
    },

    "PhoneService": {
        "Yes",
        "No"
    },

    "MultipleLines": {
        "Yes",
        "No",
        "No phone service"
    },

    "InternetService": {
        "DSL",
        "Fiber optic",
        "No"
    },

    "OnlineSecurity": {
        "Yes",
        "No",
        "No internet service"
    },

    "OnlineBackup": {
        "Yes",
        "No",
        "No internet service"
    },

    "DeviceProtection": {
        "Yes",
        "No",
        "No internet service"
    },

    "TechSupport": {
        "Yes",
        "No",
        "No internet service"
    },

    "StreamingTV": {
        "Yes",
        "No",
        "No internet service"
    },

    "StreamingMovies": {
        "Yes",
        "No",
        "No internet service"
    },

    "Contract": {
        "Month-to-month",
        "One year",
        "Two year"
    },

    "PaperlessBilling": {
        "Yes",
        "No"
    },

    "PaymentMethod": {
        "Electronic check",
        "Mailed check",
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    },

    "Churn": {
        "Yes",
        "No"
    }
}


print("\nInvalid categorical values:")

for column, valid_values in expected_categories.items():

    if column in df.columns:

        actual_values = set(
            df[column].dropna().unique()
        )

        invalid_values = actual_values - valid_values

        print(
            f"{column}:",
            invalid_values if invalid_values else "None"
        )



Invalid categorical values:
gender: None
Partner: None
Dependents: None
PhoneService: None
MultipleLines: None
InternetService: None
OnlineSecurity: None
OnlineBackup: None
DeviceProtection: None
TechSupport: None
StreamingTV: None
StreamingMovies: None
Contract: None
PaperlessBilling: None
PaymentMethod: None
Churn: None


In [81]:
print("\nData types before final verification:")
print(df.dtypes)


# Ensure numerical columns are numeric
for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


Data types before final verification:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


In [82]:
# After whitespace normalization, check again
duplicate_count_after_cleaning = df.duplicated().sum()

print(
    "\nDuplicates after cleaning:",
    duplicate_count_after_cleaning
)

if duplicate_count_after_cleaning > 0:
    df = df.drop_duplicates().copy()

print(
    "Final duplicates:",
    df.duplicated().sum()
)


Duplicates after cleaning: 0
Final duplicates: 0


In [83]:
print("\nFinal unique categorical values:")

for column in expected_categories.keys():

    if column in df.columns:

        print(f"\n{column}:")
        print(df[column].dropna().unique())


Final unique categorical values:

gender:
<ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str

Partner:
<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents:
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService:
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines:
<ArrowStringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService:
<ArrowStringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

OnlineSecurity:
<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

OnlineBackup:
<ArrowStringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

DeviceProtection:
<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

TechSupport:
<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

StreamingTV:
<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

StreamingMovies:
<ArrowStringArray>
['No', 'Y

In [84]:
print("\nFinal Missing Values:")
print(df.isnull().sum())


print("\nFinal Missing Percentage:")

final_missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

print(
    final_missing_percentage[
        final_missing_percentage > 0
    ].round(2)
)



Final Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Final Missing Percentage:
Series([], dtype: float64)


In [85]:
if "customerID" in df.columns:

    df = df.drop(
        columns=["customerID"]
    )


print("\nShape after removing customerID:")
print(df.shape)


Shape after removing customerID:
(7043, 20)


In [86]:
print("\nTarget column:")
print(df["Churn"].value_counts())

print("\nTarget unique values:")
print(df["Churn"].unique())


Target column:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Target unique values:
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [87]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [88]:
df[df["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges"]]

,tenure,MonthlyCharges,TotalCharges


In [89]:
df.loc[df["TotalCharges"].isna(), "TotalCharges"] = 0

In [90]:
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

Missing TotalCharges: 0


In [91]:
df[df["tenure"] == 0][["tenure", "MonthlyCharges", "TotalCharges"]]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,0.0
753,0,20.25,0.0
936,0,80.85,0.0
1082,0,25.75,0.0
1340,0,56.05,0.0
3331,0,19.85,0.0
3826,0,25.35,0.0
4380,0,20.00,0.0
5218,0,19.70,0.0
6670,0,73.35,0.0


In [92]:
# ============================================================
# FINAL CLEAN DATASET
# ============================================================

clean_dataset = df.copy()

print("Final Shape:", clean_dataset.shape)

print("\nMissing Values:")
print(clean_dataset.isnull().sum())

print("\nDuplicate Rows:")
print(clean_dataset.duplicated().sum())

print("\nData Types:")
print(clean_dataset.dtypes)

print("\nChurn:")
print(clean_dataset["Churn"].value_counts())

print("\nChurn Categories:")
print(clean_dataset["Churn"].unique())

Final Shape: (7043, 20)

Missing Values:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Duplicate Rows:
22

Data Types:
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           s

In [93]:
clean_dataset.to_csv(
    "../data/Clean_dataset.csv",
    index=False
)

print("Clean dataset saved successfully!")

Clean dataset saved successfully!
